In [ ]:
!pip install -q unsloth transformers==4.55.4 flask flask-cors pyngrok

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer
from threading import Thread
import json

print("Loading model with Unsloth...")

model_name = "truongduc0110/diagram-bot-8b_v4"
max_seq_length = 2048

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Chuyển sang inference mode
FastLanguageModel.for_inference(model)

# Cấu hình tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

print("✅ Model loaded successfully with Unsloth!")
print(f"📊 Model type: {type(model)}")

In [ ]:
alpaca_prompt = """
Bạn là trợ lý thiết kế database.

Nhận vào: JSON schema hiện tại + yêu cầu tự nhiên của người dùng.
Trả về: các thay đổi dưới dạng JSON gồm 4 phần:

create: []
delete: []
tomtat: "..."

Quy tắc:
- Chỉ trả về 4 trường trên, không thêm text khác.
- Các trường phải là JSON hợp lệ.
- create/update/delete là các mảng gồm các object bảng với: name, is_new, attrs[], fks[].
- Chỉ đưa ra thay đổi, không lặp lại bảng không bị chỉnh.
- tomtat ghi ngắn gọn bằng tiếng Việt.
- nếu người dùng hỏi câu hỏi không liên quan thì trả lời tôi chỉ là chatbot csdl
### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

In [ ]:
print(alpaca_prompt)

In [ ]:
import json
from flask import Flask, request, jsonify, Response
from flask_cors import CORS
from threading import Thread
import time

app = Flask(__name__)

# CORS
CORS(
    app,
    origins="*",
    supports_credentials=True,
    methods=["GET", "POST", "OPTIONS"],
    allow_headers="*"
)

@app.route("/", methods=["GET"])
def health_check():
    return jsonify({
        "status": "ok", 
        "message": "Kaggle API is running (no Unsloth)",
        "model": model_name
    })

@app.route("/generate", methods=["POST"])
def generate():
    """Non-streaming endpoint"""
    try:
        data = request.get_json()

        diagram = data.get("diagram", {})
        question = data.get("question", "")
        history = data.get("history", "None")
        max_new_tokens = data.get("max_new_tokens", 512)
        do_sample = data.get("do_sample", False)
        temperature = data.get("temperature", 0.7)
        top_p = data.get("top_p", 0.9)

        json_compact = json.dumps(diagram, ensure_ascii=False, separators=(', ', ': '))
        input_block = f"""database hiện tại: {json_compact}câu hỏi: {question}lịch sử: {history}"""

        prompt = alpaca_prompt.format("Generate edit", input_block, "") + EOS_TOKEN

        print(f"[REQUEST] {prompt}")

        inputs = tokenizer([prompt], return_tensors="pt", padding=True).to(model.device)

        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=do_sample,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        if do_sample:
            gen_kwargs.update(dict(temperature=temperature, top_p=top_p))

        with torch.inference_mode():
            outputs = model.generate(**inputs, **gen_kwargs)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        if "### Response:" in decoded:
            response_text = decoded.split("### Response:", 1)[1].strip()
        else:
            response_text = decoded.strip()

        print(f"[RESPONSE] {response_text[:100]}...")

        return jsonify({"output": response_text})
        
    except Exception as e:
        print(f"[ERROR] {str(e)}")
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

@app.route("/generate-stream", methods=["POST"])
def generate_stream():
    """Streaming endpoint với Thread"""
    try:
        data = request.get_json()

        diagram = data.get("diagram", {})
        question = data.get("question", "")
        history = data.get("history", "")
        max_new_tokens = data.get("max_new_tokens", 512)
        do_sample = data.get("do_sample", False)
        temperature = data.get("temperature", 0.7)
        top_p = data.get("top_p", 0.9)

        json_compact = json.dumps(diagram, ensure_ascii=False, separators=(',', ':'))
        input_block = f"""database hiện tại: {json_compact}câu hỏi: {question}lịch sử: {history}"""
        
        prompt = alpaca_prompt.format("Generate edit", input_block, "")

        print(f"[STREAM REQUEST] {prompt}")

        inputs = tokenizer([prompt], return_tensors="pt", padding=True).to(model.device)

        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=do_sample,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        if do_sample:
            gen_kwargs.update(dict(temperature=temperature, top_p=top_p))

        def stream():
            """Stream với Thread để tránh blocking"""
            streamer = TextIteratorStreamer(
                tokenizer,
                skip_prompt=True,
                skip_special_tokens=True
            )

            # Chạy generate trong thread riêng
            generation_kwargs = {**inputs, **gen_kwargs, "streamer": streamer}
            thread = Thread(target=model.generate, kwargs=generation_kwargs)
            thread.start()

            # Stream từng token
            for chunk in streamer:
                if chunk.strip() == "":
                    continue
                print(chunk, end="", flush=True)
                yield f"data: {json.dumps({'text': chunk}, ensure_ascii=False)}\n\n"

            print()  # Newline
            yield "data: [DONE]\n\n"
            
            thread.join()

        return Response(stream(), mimetype="text/event-stream")
        
    except Exception as e:
        print(f"[STREAM ERROR] {str(e)}")
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

In [ ]:
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

# Chạy Flask trong thread
def run_flask():
    app.run(host="0.0.0.0", port=8007, debug=False, threaded=True)

flask_thread = Thread(target=run_flask, daemon=True)
flask_thread.start()

time.sleep(3)

# Cấu hình ngrok
NGROK_TOKEN = "2nrocIUQAqRkaryy8BaerBU66NS_4zMZEgEGcwp6HE7T3H3TQ"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8007, "http").public_url

print("=" * 70)
print(f"📡 Ngrok URL: {public_url}")

In [ ]:
from pyngrok import ngrok

# Xem các tunnel đang chạy
print(ngrok.get_tunnels())

# # Tắt tất cả tunnel
ngrok.kill()


In [ ]:
text = ''':// create: [{"name":"Users","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"username","type":"VARCHAR"},{"name":"email","type":"VARCHAR"},{"name":"password","type":"VARCHAR"},{"name":"phone","type":"VARCHAR"},{"name":"role","type":"VARCHAR"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[]},{"name":"Schools","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"name","type":"VARCHAR"},{"name":"address","type":"TEXT"},{"name":"phone","type":"VARCHAR"},{"name":"email","type":"VARCHAR"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[]},{"name":"Classes","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"school_id","type":"INT"},{"name":"class_name","type":"VARCHAR"},{"name":"room_number","type":"VARCHAR"},{"name":"grade","type":"INT"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[{"column":"school_id","references":"Schools.id"}]},{"name":"Students","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"user_id","type":"INT"},{"name":"school_id","type":"INT"},{"name":"class_id","type":"INT"},{"name":"full_name","type":"VARCHAR"},{"name":"birth_date","type":"DATE"},{"name":"gender","type":"VARCHAR"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[{"column":"user_id","references":"Users.id"},{"column":"school_id","references":"Schools.id"},{"column":"class_id","references":"Classes.id"}]},{"name":"Teachers","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"user_id","type":"INT"},{"name":"school_id","type":"INT"},{"name":"teacher_code","type":"VARCHAR"},{"name":"full_name","type":"VARCHAR"},{"name":"position","type":"VARCHAR"},{"name":"phone","type":"VARCHAR"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[{"column":"user_id","references":"Users.id"},{"column":"school_id","references":"Schools.id"}]},{"name":"Subjects","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"subject_name","type":"VARCHAR"},{"name":"description","type":"TEXT"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[]},{"name":"Schedules","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"class_id","type":"INT"},{"name":"teacher_id","type":"INT"},{"name":"subject_id","type":"INT"},{"name":"day_of_week","type":"VARCHAR"},{"name":"start_time","type":"TIME"},{"name":"end_time","type":"TIME"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[{"column":"class_id","references":"Classes.id"},{"column":"teacher_id","references":"Teachers.id"},{"column":"subject_id","references":"Subjects.id"}]},{"name":"AttendanceRecords","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"student_id","type":"INT"},{"name":"class_id","type":"INT"},{"name":"attendance_date","type":"DATE"},{"name":"status","type":"VARCHAR"},{"name":"notes","type":"TEXT"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[{"column":"student_id","references":"Students.id"},{"column":"class_id","references":"Classes.id"}]},{"name":"GradeBooks","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"student_id","type":"INT"},{"name":"class_id","type":"INT"},{"name":"subject_id","type":"INT"},{"name":"grade_book_name","type":"VARCHAR"},{"name":"description","type":"TEXT"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[{"column":"student_id","references":"Students.id"},{"column":"class_id","references":"Classes.id"},{"column":"subject_id","references":"Subjects.id"}]},{"name":"Assignments","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"grade_book_id","type":"INT"},{"name":"assignment_name","type":"VARCHAR"},{"name":"description","type":"TEXT"},{"name":"due_date","type":"DATE"},{"name":"status","type":"VARCHAR"},{"name":"created_at","type":"TIMESTAMP"}],"fks":[{"column":"grade_book_id","references":"GradeBooks.id"}]},{"name":"StudentAssignmentSubmissions","is_new":true,"attrs":[{"name":"id","type":"INT","pk":true},{"name":"student_id","type":"INT"},{"name":"assignment_id","type":"INT"},{"name":"submission_date","type":"TIMESTAMP"},{"name":"file_url","type":"VARCHAR"},{"name":"status","type":"VARCHAR"},{"name":"notes","type":"TEXT"}],"fks":[{"column":"student_id","references":"Students.id"},{"column":"assignment_id","references":"Assignments.id"}]}] delete: [] tomtat: Đã tạo mới database quản lý trường học với 11 bảng: Users (quản lý người dùng), Schools (trường), Classes (lớp), Students (sinh viên), Teachers (giáo viên), Subjects (môn học), Schedules (lịch sử sử dụng), AttendanceRecords (điểm danh), GradeBooks (sổ sách học phần), Assignments (đã thi), và StudentAssignmentSubmissions (đã nộp bài). Các bảng đã được liên kết với nhau thông qua khóa ngoại để đảm bảo tính toàn vẹn dữ liệu.
'''
# Convert text thành token ids
tokens = tokenizer.tokenize(text)

print("Số token:", len(tokens))

In [ ]:
d = model.config.hidden_size
print(d)